# Day3_17 · 분류 결과와 오류 확인

실내 재실 예측의 혼동행렬과 Precision·Recall을 확인하고 로지스틱 회귀와 랜덤포레스트를 비교합니다.

## 이 노트북에서 확인할 내용

- 혼동행렬의 네 칸을 실제 상황과 연결할 수 있습니다.
- Accuracy·Precision·Recall·F1의 질문을 구분할 수 있습니다.
- 기준값과 모형이 바뀔 때 오류가 어떻게 달라지는지 비교할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-17-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-17-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-17-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-17-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-17-05 · pandas 불러오기

In [ ]:
import pandas as pd

### D3B-17-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### D3B-17-07 · 실내 재실자료 읽기

In [ ]:
occupancy = pd.read_csv(DATA_DIR / 'occupancy_detection.csv')

In [ ]:
occupancy.head()

### D3B-17-08 · 자료 크기 확인

In [ ]:
occupancy.shape

### D3B-17-09 · 열 이름 확인

In [ ]:
occupancy.columns

### D3B-17-10 · 날짜와 시간을 날짜 형식으로 바꾸기

In [ ]:
occupancy['date_time'] = pd.to_datetime(occupancy['date_time'])

In [ ]:
occupancy[['date_time']].head()

### D3B-17-11 · 학습·평가 구분 확인

In [ ]:
occupancy['source_split'].value_counts()

### D3B-17-12 · 빈 방과 재실 행 수 확인

In [ ]:
occupancy['occupied'].value_counts().sort_index()

### D3B-17-13 · 입력 센서 열 정하기

In [ ]:
sensor_columns = ['temperature_c', 'humidity_percent', 'light_lux', 'co2_ppm', 'humidity_ratio']

In [ ]:
pd.Series(sensor_columns)

### D3B-17-14 · 학습 행 선택

In [ ]:
occupancy_train = occupancy.loc[occupancy['source_split'] == 'train'].copy()

In [ ]:
occupancy_train.shape

### D3B-17-15 · 평가 행 선택

In [ ]:
occupancy_test = occupancy.loc[occupancy['source_split'] != 'train'].copy()

In [ ]:
occupancy_test.shape

### D3B-17-16 · 학습 입력과 결과 만들기

In [ ]:
X_train = occupancy_train[sensor_columns]
y_train = occupancy_train['occupied']

In [ ]:
pd.Series({'X_rows': len(X_train), 'y_rows': len(y_train)})

### D3B-17-17 · 평가 입력과 결과 만들기

In [ ]:
X_test = occupancy_test[sensor_columns]
y_test = occupancy_test['occupied']

In [ ]:
pd.Series({'X_rows': len(X_test), 'y_rows': len(y_test)})

### D3B-17-18 · 표준화와 로지스틱 회귀 불러오기

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

### D3B-17-19 · 학습자료에서 변환 규칙 익히기

In [ ]:
scaler = StandardScaler().fit(X_train)

In [ ]:
scaler

### D3B-17-20 · 학습 입력값 변환하기

In [ ]:
X_train_scaled = scaler.transform(X_train)

In [ ]:
pd.DataFrame(X_train_scaled, columns=sensor_columns).head()

### D3B-17-21 · 평가 입력값 변환하기

In [ ]:
X_test_scaled = scaler.transform(X_test)

In [ ]:
pd.DataFrame(X_test_scaled, columns=sensor_columns).head()

### D3B-17-22 · 로지스틱 회귀 학습하기

In [ ]:
logistic_model = LogisticRegression(max_iter=2000, random_state=42).fit(X_train_scaled, y_train)

In [ ]:
logistic_model

### D3B-17-23 · 평가자료 재실 확률 계산

In [ ]:
logistic_probability = logistic_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
pd.Series(logistic_probability[:10], name='probability_occupied')

### D3B-17-24 · 0.5 기준의 최종 판단 만들기

In [ ]:
logistic_prediction = (logistic_probability >= 0.5).astype(int)

In [ ]:
pd.Series(logistic_prediction).value_counts().sort_index()

## 네 종류의 판단 결과 세기

실제 재실을 맞힌 경우, 실제 빈 방을 맞힌 경우, 빈 방을 재실로 잘못 판단한 경우, 재실을 놓친 경우를 나누어 셉니다.

### D3B-17-25 · 분류 평가 도구 불러오기

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

### D3B-17-26 · 혼동행렬 계산

In [ ]:
logistic_confusion = confusion_matrix(y_test, logistic_prediction)

In [ ]:
logistic_confusion

### D3B-17-27 · 혼동행렬에 이름 붙이기

In [ ]:
confusion_table = pd.DataFrame(logistic_confusion, index=['actual empty', 'actual occupied'], columns=['predicted empty', 'predicted occupied'])

In [ ]:
confusion_table

### D3B-17-28 · 혼동행렬을 색으로 보기

In [ ]:
sns.heatmap(confusion_table, annot=True, fmt='d', cmap='Blues')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

### D3B-17-29 · 네 평가값 계산

In [ ]:
logistic_metrics = pd.Series({
    'accuracy': accuracy_score(y_test, logistic_prediction),
    'precision': precision_score(y_test, logistic_prediction),
    'recall': recall_score(y_test, logistic_prediction),
    'F1': f1_score(y_test, logistic_prediction),
})

In [ ]:
logistic_metrics.round(3)

Precision은 재실이라고 판단한 행 중 실제 재실의 비율이고, Recall은 실제 재실 행 중 모델이 찾아낸 비율입니다. 어느 오류가 더 중요한지에 따라 기준값과 평가값 선택이 달라집니다.

### D3B-17-30 · 모두 빈 방이라고 판단한 결과 만들기

In [ ]:
all_empty_prediction = pd.Series(0, index=y_test.index)

In [ ]:
all_empty_prediction.value_counts()

### D3B-17-31 · 단순 판단의 Accuracy와 Recall 확인

In [ ]:
all_empty_metrics = pd.Series({
    'accuracy': accuracy_score(y_test, all_empty_prediction),
    'recall': recall_score(y_test, all_empty_prediction, zero_division=0),
})

In [ ]:
all_empty_metrics.round(3)

Accuracy 하나만 보면 재실을 한 건도 찾지 못한 판단의 문제를 놓칠 수 있습니다. 실제 목적에 필요한 오류를 혼동행렬에서 먼저 확인합니다.

### D3B-17-32 · 기준값별 평가 함수 만들기

In [ ]:
def metrics_at_threshold(probability, actual, threshold):
    prediction = (probability >= threshold).astype(int)
    return {
        'threshold': threshold,
        'precision': precision_score(actual, prediction, zero_division=0),
        'recall': recall_score(actual, prediction, zero_division=0),
        'F1': f1_score(actual, prediction, zero_division=0),
    }

### D3B-17-33 · 여러 기준값의 평가표 만들기

In [ ]:
threshold_metrics = pd.DataFrame([metrics_at_threshold(logistic_probability, y_test, threshold) for threshold in [0.2, 0.3, 0.5, 0.7, 0.8]])

In [ ]:
threshold_metrics.round(3)

### D3B-17-34 · Precision-Recall 곡선 도구 불러오기

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

### D3B-17-35 · 곡선 좌표 계산

In [ ]:
curve_precision, curve_recall, curve_threshold = precision_recall_curve(y_test, logistic_probability)

In [ ]:
pd.DataFrame({'precision': curve_precision[:5], 'recall': curve_recall[:5]})

### D3B-17-36 · Average Precision 계산

In [ ]:
average_precision = average_precision_score(y_test, logistic_probability)

In [ ]:
average_precision

### D3B-17-37 · Precision-Recall 곡선 그리기

In [ ]:
plt.plot(curve_recall, curve_precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title(f'Precision-Recall curve (AP={average_precision:.3f})')
plt.show()

## 랜덤포레스트 분류 결과와 같은 기준으로 비교하기

### D3B-17-38 · 랜덤포레스트 분류모형 불러오기

In [ ]:
from sklearn.ensemble import RandomForestClassifier

### D3B-17-39 · 랜덤포레스트 학습하기

In [ ]:
forest_classifier = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, random_state=42, n_jobs=-1).fit(X_train, y_train)

In [ ]:
forest_classifier

### D3B-17-40 · 랜덤포레스트 재실 확률 계산

In [ ]:
forest_probability = forest_classifier.predict_proba(X_test)[:, 1]

In [ ]:
pd.Series(forest_probability[:10], name='probability_occupied')

### D3B-17-41 · 랜덤포레스트 최종 판단 만들기

In [ ]:
forest_prediction = (forest_probability >= 0.5).astype(int)

In [ ]:
pd.Series(forest_prediction).value_counts().sort_index()

### D3B-17-42 · 두 모형 평가표 만들기

In [ ]:
model_comparison = pd.DataFrame({
    'Logistic Regression': logistic_metrics,
    'Random Forest': pd.Series({
        'accuracy': accuracy_score(y_test, forest_prediction),
        'precision': precision_score(y_test, forest_prediction),
        'recall': recall_score(y_test, forest_prediction),
        'F1': f1_score(y_test, forest_prediction),
    }),
}).T

In [ ]:
model_comparison.round(3)

### D3B-17-43 · 로지스틱 회귀의 놓친 재실 행 찾기

In [ ]:
missed_occupied = occupancy_test.loc[(y_test == 1) & (logistic_prediction == 0), ['date_time'] + sensor_columns]

In [ ]:
missed_occupied.head(10).round(3)

### D3B-17-44 · 바꾸어 볼 기준값 정하기

In [ ]:
# 0.5를 0.3 또는 0.7로 바꾸어 보세요.
chosen_threshold = 0.3

In [ ]:
chosen_threshold

### D3B-17-45 · 선택한 기준값의 평가결과 계산

In [ ]:
chosen_metrics = pd.Series(metrics_at_threshold(logistic_probability, y_test, chosen_threshold))

In [ ]:
chosen_metrics.round(3)

### D3B-17-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''logistic_probability와 y_test를 사용하세요. 기존 셀은 수정하지 마세요. 기준값 0.1부터 0.9까지 Precision, Recall, F1을 계산해 표와 선그래프로 보여 주는 새 셀을 제안해 주세요. 어떤 오류를 더 줄일지 먼저 정해야 기준값을 선택할 수 있다는 설명을 포함해 주세요.'''.strip()

In [ ]:
codex_request

### D3B-17-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['평가자료만 사용하는가?', 'Precision과 Recall의 분모를 바꾸지 않는가?', '가장 높은 Accuracy만으로 기준값을 고르지 않는가?'], name='확인할 내용')

In [ ]:
codex_checklist